# Negative Selection Execution Workflow 

Verification: The notebook prints the full prompt to verify data accuracy before execution.

Execution: The task is submitted as a Slurm sbatch job. A metadata JSON file is used to orient and distribute the rows efficiently.

Hardware & Speed: The job runs on 20 parallel GPUs, taking only 3 to 4 minutes to process approximately 8,000 job ads.

Monitoring: A dedicated cell in the notebook monitors the real-time progress of the sbatch execution.

Auditing and Reporting: the notebook's audit mode automatically generates several evaluation files inside the reports subfolder to track performance and anomalies.


In [30]:
import numpy as np
import random

# =============================================================================
# EXTRACTED HELPER FUNCTIONS
# =============================================================================
def normalise_tasks(cur_tasks) -> str:
    if isinstance(cur_tasks, (list, np.ndarray)):
        xs = [str(t).strip() for t in cur_tasks if t is not None and str(t).strip()]
        return ", ".join(xs)[:800]
    return str(cur_tasks).strip()[:800]

def build_prompt(tasks_str: str, clean_titles: list, domain: str, job_ad_title: str, job_sector_category: str, full_ad_text: str) -> str:
    n_candidates = len(clean_titles)
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(clean_titles))

    full_excerpt = (full_ad_text or "").strip()[:700]
    full_block = (
        f"\nFULL AD EXCERPT (tools/duties evidence only):\n{full_excerpt}\n"
        if full_excerpt
        else ""
    )

    return f"""<|im_start|>system
Return ONLY a valid JSON object with exactly one key "drop". No extra text.
<|im_end|>

<|im_start|>user
Return ONLY a valid JSON object with exactly one key "drop". No prose. No markdown.

TASK
Audit occupation matches. DROP candidates that are NOT functional matches for the job.

KEEP POLICY
There are {n_candidates} candidates (1-based).
- Default: KEEP 2–3 candidates.
- KEEP 1 ONLY if you are certain every other candidate is clearly wrong.
- If more than 3 are valid, drop the most generic ones to fit the 3-candidate cap.
- When in doubt, KEEP rather than DROP if functionally plausible.

RANKING NOTE
Candidates are ranked by cosine similarity (rank 1 = highest cosine). This is a weak prior.
Do NOT assume rank 1 is correct.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}
{full_block}
CANDIDATES (1-based)
{numbered}

ANCHOR (FUNCTION FIRST)
- Identify the functional anchor from title + tasks (function, not seniority).
- You MUST keep the anchor unless core evidence contradicts it.
- Title keyword lock: if the title contains a clear functional keyword and a direct-match candidate exists, you MUST keep it.

GATES
- Manager rule (staff-only): if title does NOT include Manager/Lead/Director/Head/Supervisor, DROP manager/supervisor occupations unless tasks explicitly mention managing STAFF (team, colleagues, direct reports) OR rotas/shifts OR hiring/recruitment OR training/performance reviews OR budgeting/P&L.
  Notes: "manage prisoners/customers/stock/projects" is NOT staff management unless it also mentions managing a team.
  Frontline preference: if a clear frontline occupation matching the title exists (e.g., cleaner, cook, retail salesperson, warehouse worker, builder), prefer it and DROP manager/supervisor roles unless the title itself includes Manager/Lead/Supervisor.

- Inspector rule: DROP inspector/auditor/compliance occupations unless tasks explicitly mention inspection/audit/compliance checks, certification, regulatory reporting, site inspections, or audits.

- IT lock: if title/tasks mention concrete tech (Python/SQL/APIs/systems/cloud), keep relevant IT roles even if domain is non-IT.

FINAL CHECK
- Keep 2–3 by default.
- Keeping only 1 requires clear mismatch for all others.

OUTPUT
Return ONLY JSON: {{"drop":[...]}}
<|im_end|>

<|im_start|>assistant"""

# =============================================================================
# RUN LOGIC
# =============================================================================
# Pointing to the file you just built in the previous step
NPZ_PATH = "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large/adzuna_month14.npz"

print(f"Loading {NPZ_PATH}...")
data = np.load(NPZ_PATH, allow_pickle=True)

n_rows = len(data["job_ids"])
idx = random.randint(0, n_rows - 1)

# Extract row data
jid = str(data["job_ids"][idx])
titles_raw = list(data["titles"][idx]) if data["titles"][idx] is not None else []
clean_titles = [str(t).strip() for t in titles_raw if t is not None and str(t).strip()]
tasks_str = normalise_tasks(data["job_tasks"][idx])
domain = str(data["domain"][idx])
job_ad_title = str(data["job_ad_title"][idx])
job_sector_category = str(data["job_sector_category"][idx])
full_ad_text = str(data["job_description"][idx]) if data["job_description"][idx] is not None else ""

# Build and print
prompt = build_prompt(tasks_str, clean_titles, domain, job_ad_title, job_sector_category, full_ad_text)

print("\n" + "="*80)
print(f"JOB ID: {jid} (Random Row {idx} out of {n_rows:,})")
print("="*80 + "\n")
print(prompt)
print("\n" + "="*80)

Loading /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large/adzuna_month14.npz...

JOB ID: 3191523072 (Random Row 553 out of 8,426)

<|im_start|>system
Return ONLY a valid JSON object with exactly one key "drop". No extra text.
<|im_end|>

<|im_start|>user
Return ONLY a valid JSON object with exactly one key "drop". No prose. No markdown.

TASK
Audit occupation matches. DROP candidates that are NOT functional matches for the job.

KEEP POLICY
There are 7 candidates (1-based).
- Default: KEEP 2–3 candidates.
- KEEP 1 ONLY if you are certain every other candidate is clearly wrong.
- If more than 3 are valid, drop the most generic ones to fit the 3-candidate cap.
- When in doubt, KEEP rather than DROP if functionally plausible.

RANKING NOTE
Candidates are ranked by cosine similarity (rank 1 = highest cosine). This is a weak prior.
Do NOT assume rank 1 is correct.

JOB CONTEXT (SOURCE OF TRUTH)
Title: Veterinary Surgeon

## sbatch test alrady in prod bge only for month14 for preliminary test prep

In [36]:
import subprocess, re
from pathlib import Path

SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/sbatch/vllm_1gpu_array_bge_deepseek_prod_month14_test_month_chunking.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

JOBID: 2623165


# monitoring

In [38]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
<finished>


## Report deepseek

### Production Pipeline Summary

Key Components

**The Metadata Dictionary** `aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/metadata.json`
`)

Instead of counting rows during the Slurm job (which often fails due to container mount issues), we created a JSON mapping of all 14 months.

* **Why:** Instant job startup and deterministic chunking.
* **Month 14 Status:** Confirmed at **8,426 rows**.

**The Slurm Array Script**

* **Partitioning:** Uses `SLURM_ARRAY_TASK_ID` to split the workload into 20 parallel GPUs.
* **Dynamic Range:** Pulls the total row count from the JSON and calculates `START` and `STOP` indices for each worker.
* **Production Paths:** Fully aligned to the `/prod/` directory for both inputs (`.npz`) and outputs (`.jsonl`).

How to Scale Next

To process the remaining 28 million rows (Months 01–13):

1. **Update the Selection:** Change `MONTH="adzuna_month14"` to `adzuna_month01`, etc.
2. **Increase Time:** Months with ~2.5M rows will need significantly more time (e.g., `#SBATCH --time=12:00:00`).
3. **Increase Array Size:** For the heavy months, consider using 50 or 100 array tasks (`#SBATCH --array=0-99`) to maintain speed.
---

# Audit

In [31]:
PROMPT_TEXT = r"""
Return ONLY a valid JSON object with exactly one key "drop". No extra text.
<|im_end|>

<|im_start|>user
Return ONLY a valid JSON object with exactly one key "drop". No prose. No markdown.

TASK
Audit occupation matches. DROP candidates that are NOT functional matches for the job.

KEEP POLICY
There are {n_candidates} candidates (1-based).
- Default: KEEP 2–3 candidates.
- KEEP 1 ONLY if you are certain every other candidate is clearly wrong.
- If more than 3 are valid, drop the most generic ones to fit the 3-candidate cap.
- When in doubt, KEEP rather than DROP if functionally plausible.

RANKING NOTE
Candidates are ranked by cosine similarity (rank 1 = highest cosine). This is a weak prior.
Do NOT assume rank 1 is correct.

JOB CONTEXT (SOURCE OF TRUTH)
Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}
{full_block}
CANDIDATES (1-based)
{numbered}

ANCHOR (FUNCTION FIRST)
- Identify the functional anchor from title + tasks (function, not seniority).
- You MUST keep the anchor unless core evidence contradicts it.
- Title keyword lock: if the title contains a clear functional keyword and a direct-match candidate exists, you MUST keep it.

GATES
- Manager rule (staff-only): if title does NOT include Manager/Lead/Director/Head/Supervisor, DROP manager/supervisor occupations unless tasks explicitly mention managing STAFF (team, colleagues, direct reports) OR rotas/shifts OR hiring/recruitment OR training/performance reviews OR budgeting/P&L.
  Notes: "manage prisoners/customers/stock/projects" is NOT staff management unless it also mentions managing a team.
  Frontline preference: if a clear frontline occupation matching the title exists (e.g., cleaner, cook, retail salesperson, warehouse worker, builder), prefer it and DROP manager/supervisor roles unless the title itself includes Manager/Lead/Supervisor.

- Inspector rule: DROP inspector/auditor/compliance occupations unless tasks explicitly mention inspection/audit/compliance checks, certification, regulatory reporting, site inspections, or audits.

- IT lock: if title/tasks mention concrete tech (Python/SQL/APIs/systems/cloud), keep relevant IT roles even if domain is non-IT.

FINAL CHECK
- Keep 2–3 by default.
- Keeping only 1 requires clear mismatch for all others.

OUTPUT
"""


In [32]:
from pathlib import Path
import importlib.util

# load module (point to your upgraded audit script file)
p = Path("dsbg_audit.py").resolve()  # or whatever filename you saved it as
spec = importlib.util.spec_from_file_location("auditmod", p)
auditmod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(auditmod)

# paths
jsonl_base_dir = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/deepseek/bge_large"
)
npz_base_dir = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/prod/llm_negative_selection/bge_large"
)

# run (month14 is handled automatically because jsonl_base_dir contains adzuna_monthXX folders)
out = auditmod.dsbg_build_drift_report_and_pruning_lists(
    jsonl_base_dir=jsonl_base_dir,
    npz_base_dir=npz_base_dir,
    llm_model="DeepSeek-R1-Distill-Qwen-32B",
    prompt_text=PROMPT_TEXT,
    jsonl_glob="*.jsonl",
    reports_subdir="reports/deepseek_bge_audit",
    trunc_ctx_fields=1600,
    sample_trunc=650,
    max_final_expected=3,
    random_seed=7,
    random_n_any=20,
    random_n_stratum=12,
    # rerun selection rules: drift + structural failures
    select_for_rerun_flags=("it_drift", "managerial_drift", "empty_final", "final_not_subset", "missing_context"),
)

out

[OK] Wrote:
  report:   /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_DRIFT_REPORT.txt
  summary:  /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_AUDIT_SUMMARY.json
  anomalies:/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_ANOMALIES.jsonl
  random:   /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_RANDOM_SAMPLES.jsonl
  rerun:    /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_SECOND_PASS_RERUN.jsonl
  job_ids:  /lus/lfs1aip2/projects/a

{'out_dir': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601',
 'report_txt': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_DRIFT_REPORT.txt',
 'audit_summary_json': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_AUDIT_SUMMARY.json',
 'anomalies_jsonl': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_ANOMALIES.jsonl',
 'random_samples_json': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/prod/reports/deepseek_bge_audit/20260305_133601/20260305_133601_RANDOM_SAMPLES.jsonl',
 'second_pass_rerun_jsonl': '/lus/lfs1a